# Complement · Transformació de dades amb pandas

[Obre aquest quadern a Google Colab](https://colab.research.google.com/github/mlacasa/1BATXILLERAT/blob/main/Pr%C3%A0cticaBasedeDades.ipynb)

**Matemàtiques · 1r de batxillerat**  
**Autoria: Dr. Lacasa-Cazcarra · Any 2026**

**Objectiu.** Transformar dades de format ample a llarg amb una correspondència explícita entre any i categoria.

**Abans de començar:** llistes, bucles i estadística descriptiva  
**Temps orientatiu:** 2 sessions. Hi ha activitats bàsiques i ampliacions opcionals.

**Funcionament.** Executa les cel·les en ordre, des del començament. Primer fes una predicció,
després experimenta i escriu una justificació. Els controls són opcionals: també pots
modificar els arguments de les funcions i executar-les. Les figures representen mostres
finites; les propietats infinites necessiten una demostració.

**Procedència.** Reelaboració de PràcticaBasedeDades.ipynb. El CSV original de l'Agència Tributària no era al repositori.


In [ ]:
import math
from fractions import Fraction
from decimal import Decimal, localcontext
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "font.size": 11, "figure.dpi": 100})

def explora(funcio, **rangs):
    """Controls opcionals: la mateixa funció també es pot cridar directament."""
    try:
        import ipywidgets as widgets
    except ImportError:
        print("Sense controls: executem els paràmetres per defecte. Pots canviar-los a la crida.")
        return funcio()
    panell = widgets.interactive(funcio, **rangs)
    display(panell)
    return panell


## 1. Quines dades utilitzem?
**Aquest quadern treballa per defecte amb dades sintètiques de demostració.**
No són dades fiscals reals i no permeten conclusions sobre salaris ni impostos.
L'arxiu original `data2.csv` no està disponible al repositori i no es pressuposa
l'accés a cap Drive personal. Si disposes del CSV, pots carregar-lo a la cel·la indicada.

Reproduïm l'esquema de les columnes: ID, anys per a Salary, anys amb sufix .1 per a tax,
i anys amb sufix .2 per a employees. Les magnituds simulades no tenen unitats econòmiques assignades.


In [ ]:
import pandas as pd
from pathlib import Path

def dades_demostracio(files=400, anys=range(2001, 2023), llavor=2026):
    if files < 1: raise ValueError("Cal almenys una fila.")
    rng = np.random.default_rng(llavor)
    dades = {"ID": np.arange(1, files+1)}
    for sufix, escala in (("", 10000), (".1", 2000), (".2", 500)):
        for any_ in anys:
            dades[f"{any_}{sufix}"] = rng.integers(1, escala, size=files)
    return pd.DataFrame(dades)

# Opcional: puja el CSV a Colab/Jupyter i escriu-ne la ruta. None usa dades sintètiques.
ruta_csv = None  # Exemple: "data2.csv"
if ruta_csv is None:
    dades_amples = dades_demostracio()
    origen_dades = "DADES SINTÈTIQUES DE DEMOSTRACIÓ"
else:
    dades_amples = pd.read_csv(Path(ruta_csv), dtype=str)
    origen_dades = f"Fitxer proporcionat: {Path(ruta_csv).name}"
print(origen_dades)
display(dades_amples.head())
print("Dimensions:", dades_amples.shape)


## 2. Append, extend i aplanament
`append` afegeix un únic element, que pot ser una llista; `extend` afegeix
cadascun dels elements d'una seqüència. Prediu les longituds abans d'executar.


In [ ]:
a, b = [], []
a.append([1, 2, 3])
b.extend([1, 2, 3])
print("append:", a, "longitud:", len(a))
print("extend:", b, "longitud:", len(b))
matriu = np.array([[1, 2, 3], [4, 5, 6]])
print("Ordre de files:", matriu.flatten("C"))
print("Ordre de columnes:", matriu.flatten("F"))


## 3. Transformar sense confiar en la posició de les columnes
La nova taula tindrà les columnes `rank`, `value`, `year`, `category`.
Extraurem l'any i la categoria dels noms de les columnes. Així no depenem
de tenir exactament 400 files, 22 anys o un ordre determinat.

L'opció `comes_milers=True` només és adequada si el fitxer usa comes com a
separador de milers, com en els exemples guardats al quadern original.
No l'activis si les comes representen decimals.


In [ ]:
import re

def format_llarg(ample, comes_milers=False):
    if "ID" not in ample.columns or ample["ID"].isna().any() or ample["ID"].duplicated().any():
        raise ValueError("Cal una columna ID sense valors absents ni repetits.")
    if ample.empty: raise ValueError("La taula és buida.")
    categories = {None: "Salary", "1": "tax", "2": "employees"}
    blocs, ignorades = [], []
    for columna in ample.columns:
        if columna == "ID": continue
        patro = re.fullmatch(r"(\d{4})(?:\.(\d+))?", str(columna))
        if patro is None or patro.group(2) not in categories:
            ignorades.append(str(columna)); continue
        serie = ample[columna]
        if comes_milers:
            serie = serie.astype(str).str.replace(",", "", regex=False)
        valors = pd.to_numeric(serie, errors="raise")
        if not np.isfinite(valors.to_numpy(dtype=float)).all():
            raise ValueError(f"Hi ha valors absents o no finits a {columna}.")
        blocs.append(pd.DataFrame({"rank": ample["ID"].to_numpy(), "value": valors.to_numpy(),
                                    "year": int(patro.group(1)), "category": categories[patro.group(2)]}))
    if not blocs: raise ValueError("No s'han trobat columnes d'any i categoria reconegudes.")
    llarg = pd.concat(blocs, ignore_index=True)
    if llarg.duplicated(["rank", "year", "category"]).any():
        raise ValueError("Hi ha registres repetits per identificador, any i categoria.")
    return llarg, ignorades

dades_llargues, ignorades = format_llarg(dades_amples, comes_milers=ruta_csv is not None)
print(origen_dades)
print("Columnes no utilitzades:", ignorades)
print("Dimensions del format llarg:", dades_llargues.shape)
display(dades_llargues.head())


In [ ]:
def grafic_dades(any_=2001, categoria="Salary"):
    seleccio = dades_llargues[(dades_llargues["year"] == any_) & (dades_llargues["category"] == categoria)]
    fig, ax = plt.subplots()
    ax.bar(np.arange(len(seleccio)), seleccio["value"], color="#2563eb")
    ax.set(xlabel="Posició del registre seleccionat", ylabel="Valor",
           title=f"{origen_dades}\nAny {any_} · {categoria}")
    plt.tight_layout(); plt.show()
    print("Registres seleccionats:", len(seleccio))
panell = explora(grafic_dades, any_=sorted(dades_llargues["year"].unique().tolist()),
                 categoria=sorted(dades_llargues["category"].unique().tolist()))


## 4. Exportació opcional
La cel·la següent no escriu cap fitxer fins que la descomentis. L'índex tècnic
del DataFrame no és una variable de les dades i no s'ha d'afegir al CSV.


In [ ]:
# dades_llargues.to_csv("dades_format_llarg.csv", index=False)


## Activitats
1. Justifica el nombre de registres: 400·22·3=26400 a l'exemple per defecte.
2. Genera 10 files i només dos anys. Quantes files tindrà el format llarg?
3. Reordena les columnes de la taula ampla. Comprova que l'associació any–categoria–valor es conserva.
4. Què caldria documentar abans d'interpretar dades reals: font, any, unitats, població i tractament d'absents?

<details><summary>Comprovació</summary>

Amb 10 files, dos anys i tres categories hi ha 60 registres. El nom de cada columna
conserva el seu significat encara que canviem l'ordre. Les dades sintètiques només
serveixen per estudiar la transformació, no per extreure conclusions econòmiques.
</details>


**Navegació:** [Índex i guia](README.md)
